Part 1: Data Modeling and SQL 

Business stakeholders are deeply concerned about loan defaults. 
Business Problem: We need ongoing way to track default rates for different loan grades. 

Q1) Modeling: Given the dataset, what new table or view would you create in something like BigQuery to make this metric easy to access and analyze? 

Q2) Write a brief description of the table's schema (column names, data types, and a short description for each).

Q3) Justify why you chose this design over simply querying the raw data. SQL: Write a SQL query that uses your new table or view to calculate the default rate for each grade. The default rate should be defined as COUNT('Charged Off') / COUNT('All Loans'). 


solution
Part 1 – Data Modeling and SQL

Goal
Provide business stakeholders with a clear, efficient, and reliable way to track default (“Charged Off”) rates by loan grade from Lending Club loan data.

Recommended schema (column names, data types, and a short desc):


1️⃣ Modeling — Proposed View/Table
Object name: loan_default_summary (view or materialized table in BigQuery)


🎯 Step 1. Start from the Business Question
	“Stakeholders are deeply concerned about loan defaults. They need an ongoing way to track default (charge-off) rates by loan grade.”
So the business question drives everything:
	• Metric: default rate
	• Dimension: loan grade
	• Time granularity: monthly or yearly trends
This means we only need data that helps:
	1. Identify which loans defaulted (loan_status),
	2. Group them by risk tier (grade, sub_grade),
	3. Know when they were issued (issue_d),
	4. Optionally, measure financial exposure (loan_amnt, int_rate).
Everything else (employment titles, zip code, utilization ratios, etc.) is irrelevant for this specific metric — we don’t throw it away, but we don’t bring it into this summary table.


Column Name	Data Type	Description

issue_year	INT64	Year loan was issued (derived from issue_d, e.g., 2018).

issue_month	INT64	Month loan was issued (1–12).

grade	STRING	Lending Club loan grade (A–G).

sub_grade	STRING	Sub-grade (A1…G5).

loans_count	INT64	Total number of loans in the group.

charged_off_count	INT64	Number of loans whose status contains “Charged Off”.

default_rate	FLOAT64	Calculated as charged_off_count / loans_count.

sum_loan_amnt	NUMERIC	Total loan amount in the group.

avg_loan_amnt	NUMERIC	Average loan amount in the group.

avg_int_rate	FLOAT64	Average interest rate (numeric, without % sign).

created_ts	TIMESTAMP	Timestamp when summary was created.


Why this design vs querying raw data each time

• Performance: Pre-aggregating data greatly reduces the scan cost on millions of rows in BigQuery.
• Consistency: The definition of “Charged Off” is standardized once and reused across all dashboards and analyses.
• Accessibility: Stakeholders and dashboards can directly query default_rate by grade without re-implementing the logic.
• Extensibility: Easy to add new metrics (e.g., recoveries, total payments) later.


2️⃣ SQL — Create View/Table in BigQuery

CREATE OR REPLACE VIEW `project.dataset.loan_default_summary` AS
SELECT
  EXTRACT(YEAR FROM PARSE_DATE('%b-%Y', issue_d)) AS issue_year,
  EXTRACT(MONTH FROM PARSE_DATE('%b-%Y', issue_d)) AS issue_month,
  grade,
  sub_grade,
  COUNT(*) AS loans_count,
  COUNTIF(LOWER(TRIM(loan_status)) LIKE '%charged off%') AS charged_off_count,
  SAFE_DIVIDE(
    COUNTIF(LOWER(TRIM(loan_status)) LIKE '%charged off%'),
    COUNT(*)
  ) AS default_rate,
  SUM(loan_amnt) AS sum_loan_amnt,
  AVG(loan_amnt) AS avg_loan_amnt,
  AVG(CAST(REPLACE(int_rate, '%', '') AS FLOAT64)) AS avg_int_rate,
  CURRENT_TIMESTAMP() AS created_ts
FROM `project.dataset.loans_raw`
WHERE grade IS NOT NULL
  AND issue_d IS NOT NULL
GROUP BY issue_year, issue_month, grade, sub_grade;


	3) SQL — Query Default Rate per Grade
	
SELECT
  grade,
  SUM(charged_off_count) AS charged_offs,
  SUM(loans_count) AS total_loans,
  SAFE_DIVIDE(SUM(charged_off_count), SUM(loans_count)) AS default_rate
FROM `project.dataset.loan_default_summary`
GROUP BY grade
ORDER BY default_rate DESC;